In [1]:
import os
from pathlib import Path

import mlflow
from sklearn.feature_extraction.text import TfidfVectorizer

from support_assistant.data import load_intents
from support_assistant.model import build_pipeline, score_predictions, split_intents

ROOT = Path.cwd().parents[2]
os.chdir(ROOT)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("banking77-intents")

df = load_intents("data/raw/banking77_train.csv")
print(Path.cwd().name, "|", len(df), "rows")

2026/10/07 15:34:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/07 15:34:45 INFO mlflow.store.db.utils: Updating database tables
2026/10/07 15:34:45 INFO mlflow.tracking.fluent: Experiment with name 'banking77-intents' does not exist. Creating a new experiment.


support-assistant | 10003 rows


In [2]:
train_df, val_df = split_intents(df)
pipeline = build_pipeline("logistic")

with mlflow.start_run(run_name="by-hand-single-split"):
    mlflow.log_param("classifier", "logistic")
    mlflow.log_param("evaluation","one split, seed 42")

    pipeline.fit(train_df["text"], train_df["category"])
    scores = score_predictions(val_df["category"], pipeline.predict(val_df["text"]))

    mlflow.log_metric("macro_f1", scores["macro_f1"])
    mlflow.log_metric("accuracy", scores["accuracy"])
    mlflow.sklearn.log_model(pipeline, name="model")

print(scores)

2026/10/07 15:41:00 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/mac/Desktop/2026/Bootcamp/code/support-assistant
2026/10/07 15:41:08 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/mac/Desktop/2026/Bootcamp/code/support-assistant
2026/10/07 15:41:08 INFO mlflow.utils.environment: Detected uv project at /Users/mac/Desktop/2026/Bootcamp/code/support-assistant. Attempting to export requirements via 'uv export'.
2026/10/07 15:41:08 INFO mlflow.utils.uv_utils: Exported 25 dependencies via uv
2026/10/07 15:41:08 INFO mlflow.utils.environment: Successfully exported 25 requirements from uv project. Skipping package capture based inference.
2026/10/07 15:41:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'accuracy': 0.8605, 'macro_f1': 0.858317799571448}


In [3]:
pairs = TfidfVectorizer(ngram_range=(1, 2)).fit(["my top up failed", "top up not working"])
print(list(pairs.get_feature_names_out()))

tuned = build_pipeline("logistic").set_params(tfidf__ngram_range=(1, 2), classifier__C=10.0)
print(tuned.named_steps["tfidf"].ngram_range, tuned.named_steps["classifier"].C)

['failed', 'my', 'my top', 'not', 'not working', 'top', 'top up', 'up', 'up failed', 'up not', 'working']
(1, 2) 10.0
